# CALM — Adaptive Multimodal Wildfire Monitoring

**Hệ thống AI giám sát cháy rừng thích ứng đa phương thức**, theo kiến trúc agentic (URSA) và paper:

- *Adaptive Multimodal Wildfire Monitoring with Autonomous Agentic AI* (Agentic Remote Sensing)
- Kiến trúc: **Generator → Reflector → Formalizer** (3 node), **RSEN** (Reflexive Structured Experts Network), **Evidence Evaluator** (chống hallucination), **Safety Check** trước mỗi lệnh tool.

Notebook này minh họa: Planning Agent (Table A.1), RSEN (Equations 17–22), Wildfire QA, Full Pipeline (Plan + Evaluator), và Execution Agent.

## 0. Setup — Môi trường và LLM

- **Yêu cầu:** Đặt `OPENAI_API_KEY` hoặc `OPENROUTER_API_KEY` trong file `.env` tại thư mục gốc dự án (xem `.env.example`).
- **LLM:** Dùng `calm.llm_factory.get_llm()` — ưu tiên OpenRouter nếu có key, ngược lại OpenAI. Tất cả agent dùng chung một instance.
- Chạy ô dưới trước khi chạy các demo (Planning, RSEN, QA, Pipeline).


In [3]:
# Đảm bảo có thể import calm (khi mở notebook từ thư mục calm/ hoặc myProject/)
import sys
from pathlib import Path
_root = Path.cwd()
if _root.name == "calm":
    _src = _root / "src"
else:
    _src = _root / "calm" / "src"
if _src.exists() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

# Nạp biến môi trường từ .env
from calm.utils.env_loader import load_env
load_env(_root / ".env")
if _root.name != "calm" and (_root / "calm").exists():
    load_env(_root / "calm" / ".env")
else:
    load_env()

# Tạo LLM (OpenRouter hoặc OpenAI theo key có sẵn)
from calm.llm_factory import get_llm
llm = get_llm()
# Tùy chọn: chỉ định model — get_llm(model="openai/gpt-4o-mini") hoặc get_llm(model="anthropic/claude-3.5-sonnet")
print("LLM đã sẵn sàng. Có thể chạy các ô Planning, RSEN, QA, Pipeline bên dưới.")

## 1. Planning Agent (URSA 3-node, Table A.1)

Theo paper: **Generator** tạo kế hoạch từ query → **Reflector** rà soát (clarity, completeness, feasibility, safety) và trả về `[APPROVED]` nếu ổn → **Formalizer** chuyển sang JSON `plan_steps` (tối đa `f_max` lần thử). Prompt theo Appendix A Table A.1.

In [4]:
from calm.agents.planning_agent import PlanningAgent

agent = PlanningAgent(llm=llm, config={}, n_max=3, f_max=3)
query = "Wildfire risk assessment for Amazon region next 7 days"
result = agent.invoke(query)

plan = result.get("final_output") or []
print("Approved:", result.get("approved"))
print("Số bước kế hoạch:", len(plan))
for i, step in enumerate(plan, 1):
    print(f"  Bước {i}: {step.get('step_id', '?')} | action={step.get('action')} | agent={step.get('agent')}")

Approved: True
Plan:
  1. {'step_id': 'step-1', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': {'location': 'Amazon region', 'time_range': {'start': '2023-10-09T00:00:00Z', 'end': '2023-10-16T00:00:00Z'}, 'model': None}, 'expected_output': ['Detailed weather data (temperature, humidity, wind speed)'], 'success_criteria': ['Valid weather data retrieved from NOAA or Global Forecast System in JSON or CSV format with daily updates']}
  2. {'step_id': 'step-2', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': {'location': 'Amazon region', 'time_range': {'start': None, 'end': None}, 'model': None}, 'expected_output': ['Comprehensive analysis report on fire risk factors'], 'success_criteria': ['Ground-truth data collected and analyzed alongside vegetation moisture, historical fire data, drought conditions, and human activity impacts using satellite imagery']}
  3. {'step_id': 'step-3', 'action': 'retrieve_data', 'agent': 'data_knowledge', 'parameters': 

## 2. RSEN — Reflexive Structured Experts Network (Equations 17–22, Appendix A Tables A.6–A.8)

Hai chuyên gia chạy **song song, độc lập**: **Weather Analyst** (khí tượng), **Geo Analyst** (địa lý). **Ops Coordinator** tổng hợp báo cáo và đưa ra quyết định **Plausible** hoặc **Implausible**. Memory ChromaDB (`calm_rsen_memory`) dùng cho reflexion (top-k retrieval).

In [6]:
import warnings
warnings.filterwarnings("ignore", message="TqdmWarning|IProgress|BertModel|UNEXPECTED")

from calm.agents.rsen_module import RSENModule
from calm.memory.chroma_store import ChromaMemoryStore

memory = ChromaMemoryStore(
    collection_name="calm_rsen_memory",
    persist_directory=".chroma",
    k=3,
)
rsen = RSENModule(llm=llm, memory_store=memory, k=3)

result = rsen.validate(
    prediction={"risk_level": "High", "confidence": 0.8},
    met_data={"temperature": 35.0, "humidity": 0.2, "wind_speed": 15},
    spatial_data={"fuel_type": "Shrubland", "slope": 25, "elevation": 500},
)

print("Validation:", result.get("validation_decision"))
print("Rationale:", (result.get("final_rationale") or "")[:400])

/home/hungvu/code/khanh/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 13186.41it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Validation: Plausible
Rationale: The weather report highlights extreme heat and low humidity as primary factors for fire ignition and propagation, corroborated by historical patterns of similar meteorological conditions leading to wildfires. The geospatial analysis reinforces this by indicating that shrubland, combined with a steep


## 3. Wildfire QA Agent (Evidence Evaluator, Appendix A Tables A.2–A.5)

Pipeline: **retrieve** (DataKnowledgeAgent) → **Evidence Evaluator** (đánh giá bằng chứng, ngưỡng `evidence_threshold`) → nếu đủ thì trả lời, không đủ thì kích hoạt **online search** (web/ArXiv) rồi lặp. Cơ chế chống hallucination: chỉ trả lời khi có bằng chứng hỗ trợ. Mọi lệnh tool qua **Safety Check** (URSA + mở rộng CALM).

In [7]:
import warnings
warnings.filterwarnings("ignore", category=RuntimeWarning, module="duckduckgo_search")

from calm.agents.data_knowledge_agent import DataKnowledgeAgent
from calm.agents.qa_agent import WildfireQAAgent
from calm.memory.chroma_store import ChromaMemoryStore
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 5})
memory = ChromaMemoryStore(
    collection_name="calm_qa_memory",
    persist_directory=".chroma",
    k=3,
)
tools = {"earth_engine": None, "copernicus": None, "web_search": web, "arxiv": None}
data_agent = DataKnowledgeAgent(
    llm=llm, tools=tools, memory_store=memory, config={"dedup_check": True},
)
qa = WildfireQAAgent(
    llm=llm,
    data_agent=data_agent,
    web_search_tool=web,
    memory_store=memory,
    config={"evidence_threshold": 0.65},
)

result = qa.invoke("What caused the 2023 Canadian wildfires?")
out = result.get("final_output") or {}
print("Answer:", out.get("answer", out))
print("Citations:", out.get("citations", []))
print("Confidence:", out.get("confidence"))

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7263.05it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to `ddgs`! Use `pip install ddgs` instead.
  results = list(DDGS().text(query, max_results=max_results))
/home/hungvu/code/khanh/v2/calm-/src/calm/tools/web_search.py:41: RuntimeWarning: This package (`duckduckgo_search`) has been renamed to

Answer: The 2023 Canadian wildfires were primarily caused by a combination of climate change, human activities, and natural events. Key factors include: 
1. **Climate Change**: Scientific consensus indicates that climate change has led to increasingly warmer temperatures and prolonged drought conditions across Canada. This has created an environment that is more conducive to wildfires, as hot and dry conditions can significantly increase the likelihood and intensity of such events.

2. **Human Activities**: Certain land management practices, as well as human-caused ignitions (such as campfires, discarded cigarettes, and other activities), played a role in initiating some of the wildfires. In areas where development encroaches on natural habitats, there can be an increase in human-wildfire interactions.

3. **Natural Events**: Lightning strikes are a significant natural cause of wildfires, particularly in remote areas where human activity is minimal. In 2023, there were numerous instanc

## 4. Full Pipeline — Plan + LLM-as-a-Judge (Evaluator, §5.2)

Tạo kế hoạch từ query → **Evaluator Agent** chấm chất lượng output theo 5 tiêu chí (0–100): Data-Accuracy, Explainability, Jargon-Avoidance, Redundancy-Avoidance, Citation-Quality. Đạt khi `average_score >= passing_score` (mặc định 75).

In [8]:
from calm.agents.evaluator_agent import EvaluatorAgent
from calm.agents.planning_agent import PlanningAgent

planner = PlanningAgent(llm=llm, config={})
evaluator = EvaluatorAgent(llm=llm, config={"passing_score": 75.0})

query = "Assess wildfire risk for California Central Valley next 14 days"
plan_result = planner.invoke(query)
plan = plan_result.get("final_output") or []

eval_result = evaluator.evaluate(
    response={"plan": plan, "query": query},
    query=query,
)

print("Plan steps:", len(plan))
print("Evaluation passed:", eval_result.get("passed"))
print("Average score:", eval_result.get("average_score"))
print("Scores:", eval_result.get("scores", {}))
if eval_result.get("recommendations"):
    print("Recommendations:", eval_result["recommendations"])

Plan steps: 5
Evaluation passed: False
Average score: 70.0
Scores: {'data_accuracy': 70, 'explainability': 80, 'jargon_avoidance': 80, 'redundancy_avoidance': 90, 'citation_quality': 60}


## 5. Execution Agent (URSA Code Block 2 — thực thi từng bước)

Theo paper: **Task Interpretation → Safety Check → Tool Calling → Summarization**. Mỗi bước kế hoạch có thể gọi `data_knowledge`, `prediction`, hoặc `web_search`; trước mỗi lệnh tool đều qua **Safety Check**. Ví dụ: thực thi một bước từ plan (data_knowledge hoặc web_search).

In [ ]:
from calm.agents.execution_agent import ExecutionAgent
from calm.agents.planning_agent import PlanningAgent
from calm.tools.safety_check import SafetyChecker
from calm.tools.web_search import WebSearchTool

# Tạo plan một bước mẫu (hoặc lấy từ Planning Agent)
safety = SafetyChecker(llm=llm)
web = WebSearchTool(safety_checker=safety, config={"max_news_results": 3})
tools = {"data_knowledge": None, "prediction": None, "web_search": web}
exec_agent = ExecutionAgent(llm=llm, tools=tools, safety_checker=safety, config={})

# Bước mẫu: tìm kiếm web về rủi ro cháy (Execution Agent gọi web_search qua Safety Check)
step = {
    "step_id": "step-1",
    "action": "search",
    "agent": "web_search",
    "parameters": {"query": "California wildfire risk 2024"},
}
context = {"query": "Assess wildfire risk for California next 14 days"}
result = exec_agent.execute_step(step, context)
if result.get("results"):
    for i, r in enumerate(result["results"][:3], 1):
        print(f"  {i}. {r.get('title', r)}")
else:
    print("Execution result:", result)

---

### Tóm tắt theo paper (Agentic Remote Sensing / CALM)

| Thành phần | Paper | Notebook |
|------------|--------|----------|
| Planning | URSA 3-node, Table A.1 | Ô 1 — PlanningAgent |
| RSEN | Eqs 17–22, Tables A.6–A.8 | Ô 2 — RSENModule |
| QA | Evidence Evaluator, Tables A.2–A.5 | Ô 3 — WildfireQAAgent |
| Evaluator | LLM-as-a-Judge, §5.2 | Ô 4 — EvaluatorAgent |
| Execution | Safety Check, Tool Calling | Ô 5 — ExecutionAgent |

Chạy toàn bộ pipeline từ terminal: `python main.py` hoặc `calm plan "query"`. Chi tiết: README.md.